In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

# 1. Geração de dados de 50 alunos fictícios
np.random.seed(42)
n_alunos = 50

notas = np.random.uniform(2.0, 9.5, n_alunos)
frequencia = np.random.uniform(50, 100, n_alunos)

# Ajuste para aproximar da realidade: notas menores costumam ter mais faltas
frequencia = np.clip(frequencia - (10 - notas) * 3, 40, 100)

# Regra matemática invisível para definir o histórico de abandono (com ruído aleatório)
logit = 12 - 0.8 * notas - 0.12 * frequencia
prob_real = 1 / (1 + np.exp(-logit))
abandonou = (prob_real > np.random.uniform(0, 1, n_alunos)).astype(int)

df = pd.DataFrame({
    'ID_Aluno': [f'ALU_{i:03d}' for i in range(1, n_alunos + 1)],
    'Nota_Media': np.round(notas, 1),
    'Frequencia_Pct': np.round(frequencia, 1),
    'Historico_Abandonou': abandonou
})

# 2. Treinamento e aplicação da Regressão Logística
X = df[['Nota_Media', 'Frequencia_Pct']]
y = df['Historico_Abandonou']

modelo = LogisticRegression()
modelo.fit(X, y)

# 3. Calculando a probabilidade de risco gerada pelo modelo
df['Probabilidade_Abandono_Pct'] = np.round(modelo.predict_proba(X)[:, 1] * 100, 1)

# 4. Definindo a classificação com base na linha de corte (50%)
df['Classificacao_Risco'] = df['Probabilidade_Abandono_Pct'].apply(
    lambda x: 'ALTO RISCO (Alerta)' if x >= 50 else 'Baixo Risco'
)

# Salvar no Excel
df.to_excel('simulacao_risco_abandono.xlsx', index=False)
print("Arquivo Excel gerado com sucesso!")

Arquivo Excel gerado com sucesso!


In [2]:
from sklearn.metrics import classification_report, confusion_matrix

# Gerar a matriz de confusão e o relatório completo de métricas
print("--- Matriz de Confusão ---")
print(confusion_matrix(y, modelo.predict(X)))

print("\n--- Relatório de Métricas ---")
print(classification_report(y, modelo.predict(X)))

--- Matriz de Confusão ---
[[17  2]
 [ 3 28]]

--- Relatório de Métricas ---
              precision    recall  f1-score   support

           0       0.85      0.89      0.87        19
           1       0.93      0.90      0.92        31

    accuracy                           0.90        50
   macro avg       0.89      0.90      0.89        50
weighted avg       0.90      0.90      0.90        50

